## Black-Litterman Portfolio Allocation Demo

The Black-Litterman model blends **market equilibrium returns** (derived from
market-cap weights via CAPM) with **investor views** to produce stable,
intuitive portfolio allocations.

This notebook demonstrates:
1. Fetching multi-asset price data
2. Running the model with absolute and relative views
3. Comparing prior (equilibrium) vs posterior (view-adjusted) outputs

### 1. Fetch Multi-Asset Data

In [ ]:
from TerraFin import configure, load_terrafin_config
from TerraFin.data import DataFactory
from TerraFin.data.utils.filters import date_filter


configure()
config = load_terrafin_config()

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
factory = DataFactory()

closes_matrix = []
for ticker in tickers:
    data = factory.get_market_data(ticker)
    data = date_filter(data, "2024-01-01", "2025-12-31")
    closes_matrix.append(data.close.astype(float).tolist())

print(f"{len(tickers)} assets, {len(closes_matrix[0])} trading days each")

### 2. Define Inputs

**Market caps** (approximate, in $B) drive the equilibrium weights.  
**Views** express your investment thesis.

In [ ]:
# Approximate market caps in $B (order matches tickers)
# AAPL, MSFT, GOOGL, AMZN, NVDA
market_caps = [3400, 3100, 2100, 2000, 2800]

# Views:
#   1. Absolute: "NVDA will return 20% annualized" (index 4)
#   2. Relative: "AAPL will outperform GOOGL by 5%" (index 0 vs 2)
#   3. Absolute: "AMZN will return 12%" (index 3)
view_assets = [[4], [0, 2], [3]]
view_returns = [0.20, 0.05, 0.12]
view_confidences = [0.7, 0.5, 0.6]

### 3. Run Black-Litterman Model

In [ ]:
from TerraFin.analytics.analysis.portfolio import black_litterman


result = black_litterman(
    tickers=tickers,
    closes_matrix=closes_matrix,
    market_caps=market_caps,
    view_assets=view_assets,
    view_returns=view_returns,
    view_confidences=view_confidences,
)
result

### 4. Prior vs Posterior Returns

In [ ]:
import pandas as pd


df_returns = pd.DataFrame(
    {
        "Ticker": result.tickers,
        "Equilibrium Return": [f"{r:.2%}" for r in result.prior_returns],
        "BL Posterior Return": [f"{r:.2%}" for r in result.posterior_returns],
        "Shift": [f"{p - e:+.2%}" for e, p in zip(result.prior_returns, result.posterior_returns)],
    }
)
df_returns

### 5. Prior vs Posterior Weights

In [ ]:
df_weights = pd.DataFrame(
    {
        "Ticker": result.tickers,
        "Market-Cap Weight": [f"{w:.2%}" for w in result.prior_weights],
        "BL Optimal Weight": [f"{w:.2%}" for w in result.posterior_weights],
        "Shift": [f"{p - e:+.2%}" for e, p in zip(result.prior_weights, result.posterior_weights)],
    }
)
df_weights

### 6. Visual Comparison

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Expected Returns", "Portfolio Weights"),
)

# Returns comparison
fig.add_trace(
    go.Bar(x=result.tickers, y=result.prior_returns, name="Equilibrium", marker_color="steelblue"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=result.tickers, y=result.posterior_returns, name="BL Posterior", marker_color="coral"),
    row=1,
    col=1,
)

# Weights comparison
fig.add_trace(
    go.Bar(x=result.tickers, y=result.prior_weights, name="Market-Cap", marker_color="steelblue", showlegend=False),
    row=1,
    col=2,
)
fig.add_trace(
    go.Bar(x=result.tickers, y=result.posterior_weights, name="BL Optimal", marker_color="coral", showlegend=False),
    row=1,
    col=2,
)

fig.update_layout(title="Black-Litterman: Equilibrium vs View-Adjusted", barmode="group", height=450)
fig.update_yaxes(tickformat=".1%", row=1, col=1)
fig.update_yaxes(tickformat=".1%", row=1, col=2)
fig.show()

### 7. Sensitivity: Vary View Confidence
See how posterior weights shift as confidence in the NVDA view changes from 10% to 90%.

In [ ]:
import numpy as np


confidences = np.arange(0.1, 1.0, 0.1)
weight_traces = {t: [] for t in tickers}

for c in confidences:
    r = black_litterman(
        tickers=tickers,
        closes_matrix=closes_matrix,
        market_caps=market_caps,
        view_assets=[[4]],  # only NVDA view
        view_returns=[0.20],
        view_confidences=[c],
    )
    for i, t in enumerate(tickers):
        weight_traces[t].append(r.posterior_weights[i])

fig = go.Figure()
for t in tickers:
    fig.add_trace(go.Scatter(x=confidences, y=weight_traces[t], mode="lines+markers", name=t))

fig.update_layout(
    title="Sensitivity: NVDA View Confidence vs Portfolio Weights",
    xaxis_title="Confidence in NVDA 20% View",
    yaxis_title="Portfolio Weight",
    yaxis_tickformat=".1%",
    xaxis_tickformat=".0%",
)
fig.show()